In [4]:
import pandas as pd
import sqlalchemy as db
from src.get_all_data import get_all_data
import joblib

In [2]:
engine = db.create_engine('sqlite:///../data/raw/data.db')
df = get_all_data(engine)

Searching for events

[+] US Federal Funds Rate
Skipped — next release on 2026-04-29 (not yet passed)

[+] US FOMC Statement
Skipped — next release on 2026-04-29 (not yet passed)

[+] US FOMC Press Conference
Skipped — next release on 2026-04-29 (not yet passed)

[+] US FOMC Economic Projections
Skipped — next release on 2026-06-17 (not yet passed)

[+] US Core CPI m/m
Skipped — next release on 2026-04-10 (not yet passed)

[+] US CPI m/m
Skipped — next release on 2026-04-10 (not yet passed)

[+] US CPI y/y
Skipped — next release on 2026-04-10 (not yet passed)

[+] US PPI m/m
Skipped — next release on 2026-04-14 (not yet passed)

[+] US Core PCE Price Index m/m
Skipped — next release on 2026-04-09 (not yet passed)

[+] US Non-Farm Employment Change
Skipped — next release on 2026-04-03 (not yet passed)

[+] US Unemployment Rate
Skipped — next release on 2026-04-03 (not yet passed)

[+] US Average Hourly Earnings m/m
Skipped — next release on 2026-04-03 (not yet passed)

[+] US Advance GD

In [5]:
hmm = joblib.load('../models/hmm_model.pkl')
pca = joblib.load('../models/pca.pkl')
scaler = joblib.load('../models/scaler.pkl')

X_scaled = scaler.transform(df)
X_pca    = pca.transform(X_scaled)
state_predict = hmm.predict(X_pca)

n_states = hmm.n_components

df['state'] = state_predict

In [20]:
df['spy_return'] = df['spy_close'].pct_change()
df['qqq_return'] = df['qqq_close'].pct_change()
df['^vix_return'] = df['^vix_close'].pct_change()
df['dx-y.nyb_return'] = df['dx-y.nyb_close'].pct_change()
df['gc=f_return'] = df['gc=f_close'].pct_change()

# ── REGIME RETURNS ─────
return_cols = {
    'spy_return':       'SPY',
    'qqq_return':       'QQQ',
    '^vix_return':      'VIX',
    'dx-y.nyb_return':  'DXY',
    'gc=f_return':      'Gold',
}

rows = []
for state in sorted(df['state'].unique()):
    mask = df['state'] == state
    row = {'state': state, 'count': mask.sum()}
    for col, name in return_cols.items():
        ret = df.loc[mask, col]
        row[f'{name}_mean']   = ret.mean()
        row[f'{name}_std']    = ret.std()
        row[f'{name}_sharpe'] = ret.mean() / ret.std()
    rows.append(row)

regime_profile = pd.DataFrame(rows).set_index('state')

#see all states
df_reset = regime_profile.reset_index()
df_long = df_reset.melt(id_vars='state', var_name='variable', value_name='value')

df_long[['asset', 'metric']] = df_long['variable'].str.split('_', expand=True)

df_final = df_long.pivot_table(
    index=['state', 'asset'],
    columns='metric',
    values='value'
).reset_index()

df_final.columns.name = None
df_final = df_final[['state', 'asset', 'mean', 'std', 'sharpe']]

for state, group in df_final.groupby('state'):
    print(f"\nSTATE {state}")
    print(group)


STATE 0
   state asset      mean       std    sharpe
0      0   DXY -0.000036  0.006280 -0.005788
1      0  Gold  0.000073  0.018518  0.003922
2      0   QQQ  0.004325  0.023161  0.186757
3      0   SPY  0.003404  0.020704  0.164411
4      0   VIX -0.003622  0.124353 -0.029128

STATE 1
   state asset      mean       std    sharpe
5      1   DXY  0.000082  0.008726  0.009438
6      1  Gold  0.002141  0.021334  0.100378
7      1   QQQ -0.001291  0.027516 -0.046905
8      1   SPY -0.001474  0.025043 -0.058857
9      1   VIX  0.021022  0.153211  0.137208

STATE 2
    state asset      mean       std    sharpe
10      2   DXY  0.000044  0.006921  0.006424
11      2  Gold  0.000594  0.017316  0.034288
12      2   QQQ  0.001494  0.016223  0.092065
13      2   SPY  0.001126  0.013934  0.080799
14      2   VIX  0.003950  0.093283  0.042346

STATE 3
    state asset      mean       std    sharpe
15      3   DXY  0.000258  0.006202  0.041585
16      3  Gold  0.000503  0.011989  0.041928
17      3 